In [ ]:
import base64, os, pathlib, subprocess
from google.colab import userdata

REPO_URL = "https://github.com/devlucascfarias/logos-3.git"
REPO_BRANCH = "main"
WORKDIR = "/content/logos-3"
repo = pathlib.Path(WORKDIR)

token = userdata.get("GH_TOKEN")
git = ["git"]
if token:
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git += ["-c", f"http.extraHeader=Authorization: Basic {auth}"]

if (repo / ".git").exists():
    subprocess.run(git + ["-C", WORKDIR, "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
elif repo.exists():
    raise RuntimeError(f"{WORKDIR} existe, mas não é um repositório Git")
else:
    subprocess.run(git + ["clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, WORKDIR], check=True)

token = auth = None
os.chdir(WORKDIR)
print(f"Repositório sincronizado em {os.getcwd()}")

# Qwen3-8B QLoRA em NVIDIA L4

Pipeline da receita SFT. O padrão executa o piloto isolado de 500k; etapas maiores só devem ser iniciadas depois que dados, testes e avaliação comportamental passarem.

In [ ]:
STAGE = "pilot"  # pilot, baseline, main ou agentic
RUN_TRAINING = True
FRESH_RUN = True  # não retoma um piloto anterior

# None usa os 500k configurados para a etapa pilot.
TOKEN_BUDGET = None
MAX_SOURCE_ROWS = None
MAX_STEPS = None
MAX_TRAIN_SAMPLES = None
REFERENCE_ADAPTER_PATH = "/content/drive/MyDrive/logos-3/adapters/baseline_smoke_v2"
EVAL_SEED = 20260722
print({"stage": STAGE, "token_budget_override": TOKEN_BUDGET, "fresh": FRESH_RUN})

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    login(token=hf_token, add_to_git_credential=False)
else:
    print("HF_TOKEN não definido; apenas fontes públicas sem aceite funcionarão.")
hf_token = None

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "scripts/environment_check.py"], check=True)

In [ ]:
command = [sys.executable, "scripts/prepare_data.py", "--stage", STAGE]
if TOKEN_BUDGET is not None:
    command += ["--token-budget", str(TOKEN_BUDGET)]
if MAX_SOURCE_ROWS is not None:
    command += ["--max-source-rows", str(MAX_SOURCE_ROWS)]
subprocess.run(command, check=True)

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

In [ ]:
import codecs, datetime, os, pathlib, shutil, subprocess, sys

if RUN_TRAINING:
    fresh_run = FRESH_RUN
    if fresh_run:
        timestamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        archive_root = pathlib.Path(WORKDIR) / "outputs" / "archive" / f"{STAGE}-{timestamp}"
        old_outputs = {
            "checkpoints": pathlib.Path(WORKDIR) / "outputs" / "checkpoints" / STAGE,
            "adapter": pathlib.Path(WORKDIR) / "outputs" / "adapters" / STAGE,
        }
        for label, source in old_outputs.items():
            if source.exists():
                archive_root.mkdir(parents=True, exist_ok=True)
                shutil.move(str(source), str(archive_root / label))
                print(f"Arquivado: {source} -> {archive_root / label}")
        FRESH_RUN = False
    command = [sys.executable, "-u", "scripts/train_sft.py", "--stage", STAGE]
    if not fresh_run:
        command += ["--resume-from-checkpoint", "auto"]
    if MAX_STEPS is not None:
        command += ["--max-steps", str(MAX_STEPS)]
    if MAX_TRAIN_SAMPLES is not None:
        command += ["--max-train-samples", str(MAX_TRAIN_SAMPLES)]
    print("Iniciando treino com barra de progresso e ETA...", flush=True)
    log_path = os.path.join(WORKDIR, "outputs", "logs", f"{STAGE}_train.log")
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    process = subprocess.Popen(
        command,
        cwd=WORKDIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0,
    )
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    assert process.stdout is not None
    with open(log_path, "wb") as log_file:
        while True:
            chunk = os.read(process.stdout.fileno(), 4096)
            if not chunk:
                break
            log_file.write(chunk)
            log_file.flush()
            sys.stdout.write(decoder.decode(chunk))
            sys.stdout.flush()
        sys.stdout.write(decoder.decode(b"", final=True))
        sys.stdout.flush()
    return_code = process.wait()
    print(f"\nLog salvo em: {log_path}", flush=True)
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
else:
    print("RUN_TRAINING=False: dados e testes prontos; treino não iniciado.")

## Comparação cega: base vs. campeão 250k vs. piloto 500k

Esta etapa gera respostas determinísticas para os mesmos prompts inéditos. Se o adapter campeão estiver disponível no Google Drive, as três identidades serão embaralhadas como A/B/C; avalie `comparison.md` antes de abrir `mapping.json`.

In [ ]:
import codecs, os, pathlib, subprocess, sys

if REFERENCE_ADAPTER_PATH.startswith("/content/drive/") and not pathlib.Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

EVAL_OUTPUT_DIR = "outputs/evaluations/pilot_500k"
command = [
    sys.executable, "-u", "scripts/compare_adapter.py",
    "--stage", STAGE,
    "--output-dir", EVAL_OUTPUT_DIR,
    "--seed", str(EVAL_SEED),
]
if REFERENCE_ADAPTER_PATH and pathlib.Path(REFERENCE_ADAPTER_PATH).exists():
    command += ["--reference-adapter-path", REFERENCE_ADAPTER_PATH]
else:
    print("AVISO: campeão 250k não encontrado; comparação terá apenas base e piloto.")
print("Iniciando comparação cega com progresso...", flush=True)
log_path = os.path.join(WORKDIR, "outputs", "logs", f"{STAGE}_comparison.log")
os.makedirs(os.path.dirname(log_path), exist_ok=True)
process = subprocess.Popen(
    command,
    cwd=WORKDIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=0,
)
decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
assert process.stdout is not None
with open(log_path, "wb") as log_file:
    while True:
        chunk = os.read(process.stdout.fileno(), 4096)
        if not chunk:
            break
        log_file.write(chunk)
        log_file.flush()
        sys.stdout.write(decoder.decode(chunk))
        sys.stdout.flush()
    sys.stdout.write(decoder.decode(b"", final=True))
    sys.stdout.flush()
return_code = process.wait()
print(f"\nLog salvo em: {log_path}", flush=True)
if return_code:
    raise subprocess.CalledProcessError(return_code, command)

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

comparison_path = Path(WORKDIR) / EVAL_OUTPUT_DIR / "comparison.md"
display(Markdown(comparison_path.read_text(encoding="utf-8")))
print("Avalie A/B acima antes de abrir mapping.json.")
print("Planilha de notas:", Path(WORKDIR) / EVAL_OUTPUT_DIR / "ratings.json")

## Depois da avaliação

Somente após atribuir as notas, abra `outputs/evaluations/pilot_500k/mapping.json`. O próximo treino deve ser decidido pela comparação entre base, campeão e piloto, não apenas pelo `eval_loss`.